# 05 — Size-adaptive inference: spending the latency margin

A 512×512 frame costs about **10 ms** on the Orin and roughly **20–30 ms** is
affordable. This notebook asks what that margin actually buys.

## Not SAHI as published

Slicing Aided Hyper Inference tiles a frame and runs a **detector** on each
tile. This pipeline has no detector — it has one prompted target and a memory
bank. Tiling every frame would be paying detector prices for tracker work.

The equivalent for a tracker is a **window that follows the target and sizes
itself to it**: crop a region a fixed multiple of the target's size, feed it to
the model at 512, and a 10-pixel drone arrives as a 40-pixel one. Nothing about
the model changes; it simply stops being asked to segment something smaller
than its own stride.

Tiling still earns a place — but only where a detector-shaped technique
belongs: **re-acquisition** after the target has been lost, at maybe one frame
in thirty.

## The hazard, stated before the results

SAM 2 stores memory features in **input coordinates**. Move the crop and every
stored memory is in a different frame of reference from the features reading
it. So the window is treated as a **segment boundary**:

- it holds still — a size ladder, a position grid, and a margin before either
  is touched;
- when it must move, tracking restarts at the new window with a fresh memory
  bank, re-prompted from the last mask.

That makes the coordinate change explicit rather than silently approximate. It
also means **each move costs a re-anchor**, so the number of moves is as much a
result as the accuracy is. This is the part of the whole plan most likely not
to pay for itself, and the measurement below is what decides it.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/sam-dedection")
if not REPO.exists():
    !git clone -q https://github.com/yigitkayabagci/sam-dedection.git {REPO}
os.chdir(REPO); sys.path.insert(0, str(REPO))

!bash scripts/setup_edgetam.sh 2>&1 | tail -3
!pip install -q -r requirements.txt
!python -m unittest tests.test_adaptive 2>&1 | tail -3

In [ ]:
# --- Data on local disk, keepsakes on Drive -----------------------------
# The dataset is a few hundred thousand small JPEGs; the Drive FUSE mount
# serves those an order of magnitude slower than the GPU reads them. Drive
# holds only what is worth surviving the runtime: the checkpoint and the RLE
# label store, both megabytes.
DATA_DIR = Path("/content/data")
!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits train val

from tools.fetch_antiuav410 import dataset_root, describe, find_splits

splits = find_splits(DATA_DIR)
DATA = dataset_root(splits)
print(describe(splits))

WORK = Path("/content/work")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/edgetam-thermal")
except Exception as exc:
    print(f"no Drive ({type(exc).__name__}) -- WORK stays at {WORK}")
WORK.mkdir(parents=True, exist_ok=True)

import shutil
CKPT = REPO / "checkpoints"; CKPT.mkdir(exist_ok=True)
CONFIG = "configs/edgetam_512.yaml"
if (WORK / "edgetam_thermal_512.pt").exists():
    shutil.copy(WORK / "edgetam_thermal_512.pt", CKPT / "edgetam_thermal_512.pt")
    CONFIG = "configs/edgetam_512_thermal.yaml"

## Is this worth doing on your data at all?

The whole idea rests on one premise: **targets are small enough that a fixed
512 input is throwing the signal away.** That is a measurable property of the
dataset, not an assumption, and it should be checked before anything is built
on it.

At 512, a stride-16 backbone sees a target of `n` source pixels as roughly
`n/1.25` pixels (640→512), which is under 1 feature cell for anything below
about 20 px. If most of the distribution sits above that, close this notebook —
the margin is better spent elsewhere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.training import list_sequences

sequences = list_sequences(DATA, "val")[:40]
sides = np.concatenate([
    np.nanmax(np.stack([s.labels.boxes[:, 2] - s.labels.boxes[:, 0],
                        s.labels.boxes[:, 3] - s.labels.boxes[:, 1]]), axis=0)
    for s in sequences
])
sides = sides[np.isfinite(sides)]
at_512 = sides * (512 / 640)
cells = at_512 / 16   # backbone stride

print(f"{len(sides)} targets. At 512 input, in backbone feature cells:")
for label, threshold in (("under 1 cell", 1), ("under 2 cells", 2), ("under 4 cells", 4)):
    print(f"  {label:<14} {np.mean(cells < threshold):6.1%}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(cells, bins=60, range=(0, 12), color="#2a78d6")
ax.axvline(1, color="#eb6834", lw=1.4, label="1 feature cell")
ax.set_xlabel("target size at 512 input (backbone feature cells)")
ax.set_ylabel("frames"); ax.legend()
plt.tight_layout(); plt.show()
print("\nMass to the left of the orange line is the case for this notebook: "
      "the model is being asked to segment something smaller than one cell of "
      "the feature map it reasons over.")

## Baseline, then adaptive

Same tracker, same checkpoint, same sequences. The only difference is what the
model is shown.

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit 12 --mode crop \
    --tracker edgetam --config {CONFIG} --json results/fixed.json 2>&1 | tail -18

In [ ]:
!python tools/track_adaptive.py --data {DATA} --split val --limit 12 \
    --tracker edgetam --config {CONFIG} \
    --target-fraction 0.08 --min-window 128 --scale-step 1.5 --margin 0.2 \
    --json results/adaptive.json 2>&1 | tail -20

In [ ]:
import json

def summarise(path):
    rows = json.loads(Path(path).read_text())["sequences"]
    frames = sum(r["frames"] for r in rows)
    lengths = [n for r in rows for n in r["dropout_lengths"]]
    moves = [m for r in rows for m in r.get("segments", [])
             if not m.endswith(":chunk")]
    return {
        "state accuracy": sum(r["state_accuracy"] * r["frames"] for r in rows) / frames,
        "success AUC": sum(r["success_auc"] * r["frames"] for r in rows) / frames,
        "frames lost": sum(lengths),
        "window moves": len(moves),
        "frames": frames,
    }

fixed, adaptive = summarise("results/fixed.json"), summarise("results/adaptive.json")
print(f"{'':<16}{'fixed 512':>12}{'adaptive':>12}")
for key in fixed:
    a, b = fixed[key], adaptive[key]
    fmt = "{:>12.4f}" if isinstance(a, float) else "{:>12}"
    print(f"{key:<16}" + fmt.format(a) + fmt.format(b))

per_100 = 100 * adaptive["window moves"] / max(adaptive["frames"], 1)
print(f"\n{per_100:.1f} window moves per 100 frames. Each one is a memory "
      "re-anchor -- the tracker starts again from its own last mask.")
print("More than ~2 per 100 and the re-anchors are likely costing more than "
      "the zoom is buying; raise --margin and --scale-step before concluding "
      "the idea does not work.")

## Where the trade-off actually sits

`target_fraction` is the one knob that matters. Smaller means a tighter crop,
more magnification and a better view of the target — and a window that has to
move more often, because the target crosses a smaller field faster.

There is no reason to expect the best value to be the smallest, and the sweep
is cheap compared to guessing.

In [ ]:
from tqdm.auto import tqdm

sweep = []
for fraction in tqdm([0.05, 0.08, 0.12, 0.20, 0.35], desc="target_fraction"):
    out = Path(f"results/adaptive_{fraction}.json")
    !python tools/track_adaptive.py --data {DATA} --split val --limit 6 \
        --tracker edgetam --config {CONFIG} --target-fraction {fraction} \
        --json {out} > /dev/null 2>&1
    if out.exists():
        sweep.append({"target_fraction": fraction, **summarise(out)})

print(f"{'fraction':>10}{'state acc':>12}{'frames lost':>13}{'moves/100':>11}")
for row in sweep:
    print(f"{row['target_fraction']:>10}{row['state accuracy']:>12.4f}"
          f"{row['frames lost']:>13}"
          f"{100 * row['window moves'] / max(row['frames'], 1):>11.1f}")

if sweep:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    fractions = [r["target_fraction"] for r in sweep]
    ax.plot(fractions, [r["state accuracy"] for r in sweep], "o-", color="#2a78d6")
    ax.set_xlabel("target_fraction (smaller = tighter crop)")
    ax.set_ylabel("state accuracy", color="#2a78d6")
    twin = ax.twinx()
    twin.plot(fractions, [100 * r["window moves"] / max(r["frames"], 1) for r in sweep],
              "s--", color="#eb6834")
    twin.set_ylabel("window moves per 100 frames", color="#eb6834")
    ax.axhline(fixed["state accuracy"], color="#898781", ls=":", label="fixed 512")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.show()

## The latency side

Adaptive inference does not make the model slower — the input is still 512×512
and the engines are unchanged. What it adds is per-frame **cropping** (a memory
copy, no resize when the window is already 512) and, at a boundary, one
re-prompt.

That means the budget question is not "does the model still fit in 30 ms" but
"how often is a boundary crossed". Measure it on the Orin the same way
everything else here is measured:

```bash
python tools/track_adaptive.py --data <anti-uav410> --split val --limit 20 \
    --tracker edgetam_trt --config configs/edgetam_trt_512.yaml \
    --json results/adaptive_trt.json
```

## Re-acquisition, and where tiling belongs

`src/trackers/adaptive.py` also provides `tile_grid` and `Reacquisition`: when
the target has been lost for `patience` frames there is nothing left to follow,
and a tile sweep to find it again is affordable at one frame in tens in a way
it never is every frame.

The driver does not wire that in yet — deliberately. **Check whether SAMURAI
(notebook 04) already fixed the losses first.** Its memory gate stops a bad
frame poisoning the bank, which is the most common reason a target stays lost;
if the dropout episodes are already short, a re-acquisition sweep has nothing
to re-acquire and is pure cost.

```python
from src.trackers.adaptive import Reacquisition, tile_grid
state = Reacquisition(patience=30, cooldown=30)
tiles = tile_grid((640, 512), tile=256, overlap=0.2)   # 20% overlap: a target
                                                        # on a boundary is split
                                                        # and convincing in neither
```

## What to conclude

| what you see | what it means |
|---|---|
| accuracy up, < 2 moves per 100 frames | the zoom is paying for itself — take it |
| accuracy up, many moves | the re-anchors are eating the gain; raise `--margin` and `--scale-step` and re-run before deciding |
| accuracy down | the memory bank is being invalidated faster than the magnification helps. This is the outcome to be honest about: fixed `crop512` is then the right answer, and the margin should go to SAM2Long or a larger input instead |

Note the last row is a real possibility and the reason this notebook is last.
Of everything in this plan, size-adaptive inference is the piece most likely
not to work, because it fights the memory bank rather than working with it.